# FLUX Fill BSS/BDS Condition-Anchoring Run

This notebook mounts Google Drive, clones or updates the lightweight experiment repo, downloads FLUX.1 Fill-dev weights to Drive only when missing, runs audit, creates synthetic fill assets/manifests, runs smoke first, and only runs the mini-suite when `RUN_MINI_SUITE=True`.

Do not paste Hugging Face tokens into code cells. The token prompt below uses `getpass` and does not print the token.

In [ ]:
from pathlib import Path
import os

# Set this before Run All after you push the scaffold to GitHub.
GITHUB_REPO_URL = "https://github.com/WANG-Ruipeng/FLUX-bss.git"
BRANCH = "flux-fill-bss-bds"
REPO_ROOT = Path("/content/FLUX-bss")

DRIVE_ROOT = Path("/content/drive/MyDrive/Colab_Projects/FLUX-Fill-BSS-BDS")
DRIVE_WEIGHTS_ROOT = Path("/content/drive/MyDrive/ModelWeights/FLUX/FLUX.1-Fill-dev")
EXPERIMENT_ROOT = Path("/content/FLUX-Fill-BSS-Runs/flux_fill_bss_bds_v1")
DRIVE_EXPERIMENT_ROOT = DRIVE_ROOT / "flux_fill_bss_bds_v1"

INSTALL_DEPS = True
MOUNT_DRIVE = True
FORCE_RECLONE = False

# User agreed to the gated license before running this notebook.
LICENSE_ACCEPTED = True

# If weights are missing, this notebook asks for HF token and downloads to Drive.
AUTO_DOWNLOAD_IF_MISSING = True
DOWNLOAD_WEIGHTS = False

# Keep smoke enabled. Enable mini-suite only after smoke passes.
RUN_SMOKE = True
RUN_MINI_SUITE = False

HEIGHT = 1024
WIDTH = 1024
GUIDANCE_SCALE = 30.0
MAX_SEQUENCE_LENGTH = 512
SEED = 0
DTYPE = "bfloat16"
CPU_OFFLOAD = False

print("REPO_ROOT:", REPO_ROOT)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("DRIVE_EXPERIMENT_ROOT:", DRIVE_EXPERIMENT_ROOT)
print("DRIVE_WEIGHTS_ROOT:", DRIVE_WEIGHTS_ROOT)
print("RUN_SMOKE:", RUN_SMOKE)
print("RUN_MINI_SUITE:", RUN_MINI_SUITE)


In [ ]:
import subprocess
import sys
import time
import shutil
import getpass

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Drive mount skipped or failed:", repr(exc))

EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Runtime setup done.")


In [ ]:
def run(cmd, cwd=None, check=True, env=None):
    display = " ".join(str(part) for part in cmd)
    if "x-access-token:" in display:
        display = display.split("x-access-token:")[0] + "x-access-token:***@" + display.split("@", 1)[-1]
    print("$", display)
    return subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=check, text=True, env=env)


def authenticated_url(url):
    if "<YOUR_ORG_OR_USER>" in url:
        return url
    token = os.environ.get("GITHUB_TOKEN", "")
    if not token:
        token = getpass.getpass("GitHub token for clone/fetch; leave blank if public: ")
        if token:
            os.environ["GITHUB_TOKEN"] = token
    if token:
        return url.replace("https://", f"https://x-access-token:{token}@", 1)
    return url


if FORCE_RECLONE and REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

if "<YOUR_ORG_OR_USER>" in GITHUB_REPO_URL and not REPO_ROOT.exists():
    raise RuntimeError("Set GITHUB_REPO_URL to your pushed repo before running in a fresh Colab runtime.")

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    run(["git", "-C", REPO_ROOT, "fetch", "origin", BRANCH], check=False)
    run(["git", "-C", REPO_ROOT, "checkout", BRANCH], check=False)
    run(["git", "-C", REPO_ROOT, "pull", "--ff-only", "origin", BRANCH], check=False)
elif not REPO_ROOT.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", authenticated_url(GITHUB_REPO_URL), REPO_ROOT])
    run(["git", "-C", REPO_ROOT, "remote", "set-url", "origin", GITHUB_REPO_URL], check=False)

run(["git", "-C", REPO_ROOT, "branch", "--show-current"], check=False)
run(["git", "-C", REPO_ROOT, "rev-parse", "HEAD"], check=False)
run(["git", "-C", REPO_ROOT, "status", "--short"], check=False)


In [ ]:
if INSTALL_DEPS:
    run([
        sys.executable, "-m", "pip", "install", "-U",
        "diffusers", "transformers", "accelerate", "safetensors", "sentencepiece",
        "huggingface_hub", "opencv-python", "pillow", "imageio", "pandas", "numpy", "matplotlib"
    ])


In [ ]:
def weights_ready(path):
    return path.exists() and (path / "model_index.json").exists()


need_download = DOWNLOAD_WEIGHTS or (AUTO_DOWNLOAD_IF_MISSING and not weights_ready(DRIVE_WEIGHTS_ROOT))
if need_download:
    if not LICENSE_ACCEPTED:
        raise RuntimeError("Accept the FLUX.1 Fill-dev gated license before downloading weights.")
    from huggingface_hub import login
    hf_token = os.environ.get("HF_TOKEN", "") or os.environ.get("HUGGINGFACE_HUB_TOKEN", "")
    if not hf_token:
        hf_token = getpass.getpass("Hugging Face token with accepted FLUX.1 Fill-dev access: ")
    login(token=hf_token, add_to_git_credential=False)
    DRIVE_WEIGHTS_ROOT.mkdir(parents=True, exist_ok=True)
    run([
        "huggingface-cli", "download", "black-forest-labs/FLUX.1-Fill-dev",
        "--local-dir", DRIVE_WEIGHTS_ROOT,
        "--local-dir-use-symlinks", "False"
    ])
else:
    print("Using existing Drive weights:", DRIVE_WEIGHTS_ROOT)

if not weights_ready(DRIVE_WEIGHTS_ROOT):
    raise RuntimeError(f"Weights are still missing or incomplete at {DRIVE_WEIGHTS_ROOT}")


In [ ]:
env = os.environ.copy()
env["REPO_ROOT"] = str(REPO_ROOT)
env["BSS_CONDITION_REPO_DIR"] = str(REPO_ROOT)
env["EXPERIMENT_ROOT"] = str(EXPERIMENT_ROOT)
env["DRIVE_EXPERIMENT_ROOT"] = str(DRIVE_EXPERIMENT_ROOT)
env["DRIVE_WEIGHTS_ROOT"] = str(DRIVE_WEIGHTS_ROOT)

SCRIPT_ROOT = REPO_ROOT / "bss_experiments/flux_fill_bss_bds_v1/scripts"
license_flag = "yes" if LICENSE_ACCEPTED else "unknown"
audit_cmd = [
    sys.executable, SCRIPT_ROOT / "audit_flux_fill.py",
    "--run_root", EXPERIMENT_ROOT,
    "--drive_weights_root", DRIVE_WEIGHTS_ROOT,
    "--license_accepted", license_flag,
    "--dtype", DTYPE,
]
if CPU_OFFLOAD:
    audit_cmd.append("--cpu_offload")
run(audit_cmd, cwd=REPO_ROOT, env=env)


In [ ]:
run([
    sys.executable, SCRIPT_ROOT / "make_flux_fill_assets_and_manifest.py",
    "--run_root", EXPERIMENT_ROOT,
    "--height", HEIGHT,
    "--width", WIDTH,
    "--guidance_scale", GUIDANCE_SCALE,
    "--max_sequence_length", MAX_SEQUENCE_LENGTH,
    "--seed", SEED,
    "--dtype", DTYPE,
], cwd=REPO_ROOT, env=env)

SMOKE_MANIFEST = EXPERIMENT_ROOT / "manifests/flux_fill_smoke_manifest.csv"
MINI_MANIFEST = EXPERIMENT_ROOT / "manifests/flux_fill_mini_manifest.csv"
run([sys.executable, SCRIPT_ROOT / "validate_schedule.py", "--manifest", SMOKE_MANIFEST], cwd=REPO_ROOT, env=env)
run([sys.executable, SCRIPT_ROOT / "validate_schedule.py", "--manifest", MINI_MANIFEST], cwd=REPO_ROOT, env=env)


In [ ]:
if RUN_SMOKE:
    cmd = [
        sys.executable, SCRIPT_ROOT / "run_manifest.py",
        "--manifest", SMOKE_MANIFEST,
        "--run_root", EXPERIMENT_ROOT,
        "--drive_run_root", DRIVE_EXPERIMENT_ROOT,
        "--model_path", DRIVE_WEIGHTS_ROOT,
        "--resume",
        "--sync_drive", "true",
        "--dtype", DTYPE,
    ]
    if CPU_OFFLOAD:
        cmd.append("--cpu_offload")
    run(cmd, cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "validate_schedule.py", "--manifest", SMOKE_MANIFEST, "--require_schedule_files"], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "compute_metrics_against_ref.py", "--manifest", SMOKE_MANIFEST, "--run_root", EXPERIMENT_ROOT, "--allow_missing"], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "make_figures.py", "--manifest", SMOKE_MANIFEST, "--run_root", EXPERIMENT_ROOT], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "write_run_reports.py", "--run_root", EXPERIMENT_ROOT, "--smoke_manifest", SMOKE_MANIFEST, "--mini_manifest", MINI_MANIFEST], cwd=REPO_ROOT, env=env)


In [ ]:
if RUN_MINI_SUITE:
    cmd = [
        sys.executable, SCRIPT_ROOT / "run_manifest.py",
        "--manifest", MINI_MANIFEST,
        "--run_root", EXPERIMENT_ROOT,
        "--drive_run_root", DRIVE_EXPERIMENT_ROOT,
        "--model_path", DRIVE_WEIGHTS_ROOT,
        "--resume",
        "--sync_drive", "true",
        "--dtype", DTYPE,
    ]
    if CPU_OFFLOAD:
        cmd.append("--cpu_offload")
    run(cmd, cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "validate_schedule.py", "--manifest", MINI_MANIFEST, "--require_schedule_files"], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "compute_metrics_against_ref.py", "--manifest", MINI_MANIFEST, "--run_root", EXPERIMENT_ROOT, "--allow_missing"], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "compute_bds.py", "--run_root", EXPERIMENT_ROOT, "--allow_missing"], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "make_tables_flux_fill.py", "--run_root", EXPERIMENT_ROOT], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "make_figures.py", "--manifest", MINI_MANIFEST, "--run_root", EXPERIMENT_ROOT], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "write_run_reports.py", "--run_root", EXPERIMENT_ROOT, "--smoke_manifest", SMOKE_MANIFEST, "--mini_manifest", MINI_MANIFEST], cwd=REPO_ROOT, env=env)
    run([sys.executable, SCRIPT_ROOT / "write_final_report.py", "--run_root", EXPERIMENT_ROOT, "--notebook_path", REPO_ROOT / "notebooks/flux_fill_colab_bss_bds.ipynb"], cwd=REPO_ROOT, env=env)
else:
    print("RUN_MINI_SUITE is False. Review smoke outputs before enabling the mini-suite.")
